# Molecular representation comparison

Predict an RDKit-computed property (logP) from three interchangeable encoders over the same molecules: Morgan fingerprints with an MLP, tokenized SMILES with a 1D CNN, and tokenized SMILES with an LSTM. This runs fully offline on the packaged sample via `load_sample`; `scripts/benchmark.py` runs the same comparison on a larger downloaded set.

In [1]:
from molprop import load_sample, random_split, Trainer, set_seed, available_encoders

set_seed(0)
dataset = load_sample(target='logp', n_bits=1024)
train_set, val_set, test_set = random_split(dataset, seed=0)
vocab = len(dataset.tokenizer)
print('encoders:', available_encoders())
len(train_set), len(val_set), len(test_set), vocab

encoders: ['fingerprint_mlp', 'smiles_cnn', 'smiles_rnn']


(240, 30, 30, 32)

In [2]:
from molprop import build_encoder

for name in available_encoders():
    set_seed(0)
    model = build_encoder(name, vocab_size=vocab, n_bits=1024)
    trainer = Trainer(model, lr=1e-3, weight_decay=1e-5, patience=10)
    trainer.fit(train_set, val_set, epochs=60, batch_size=16, verbose=False)
    print(name, 'best_epoch', trainer.best_epoch, trainer.evaluate(test_set))

fingerprint_mlp best_epoch 5 {'rmse': 1.0545161999328747, 'mae': 0.8182874182860057, 'r2': 0.6664876255585237}


smiles_cnn best_epoch 9 {'rmse': 0.7073248813543274, 'mae': 0.5525281846523284, 'r2': 0.849947473836671}


smiles_rnn best_epoch 30 {'rmse': 0.6196498546312396, 'mae': 0.4731911559899648, 'r2': 0.8848409311127312}
